# Map Equations 49-54
This notebook focuses on generating the rasters for equations 49 to 54 using an optimized workflow.


In [ ]:
# Core dependencies
from pathlib import Path
import numpy as np
import rasterio

In [ ]:
# User parameters
NODATA_VALUE = 255
PATH_SERIES_X = Path(r"C:\Users\AntFonseca\github\1.INPUT\compare-time-series\PIE\pixelbasedZoom2")
PATH_SERIES_Y = Path(r"C:\Users\AntFonseca\github\1.INPUT\compare-time-series\PIE\objectbasedZoom2")
TIME_POINTS = [2010, 2012, 2014, 2016, 2018, 2021]
CLASS_NAME = "PIE"
OUTPUT_PATH = Path(r"C:\Users\AntFonseca\github\2.OUTPUT\PIEzoomed2")
OUTPUT_DEFINITIONS = {
    "Equation49": {
        "dtype": np.uint8,
        "nodata": 255,
        "filename": "Equation49_result.tif",
    },
    "Equation50": {
        "dtype": np.int8,
        "nodata": -128,
        "filename": "Equation50_result.tif",
    },
    "Equation51": {
        "dtype": np.uint8,
        "nodata": 255,
        "filename": "Equation51_result.tif",
    },
    "Equation52": {
        "dtype": np.int8,
        "nodata": -128,
        "filename": "Equation52_result.tif",
    },
    "Equation53": {
        "dtype": np.int8,
        "nodata": -128,
        "filename": "Equation53_result.tif",
    },
    "Equation54": {
        "dtype": np.uint8,
        "nodata": 255,
        "filename": "Equation54_result.tif",
    },
}
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)


In [ ]:
# Helper functions

def read_stack(
    folder: Path,
    time_points: list[int],
    class_name: str,
) -> np.ndarray:
    """Load the raster stack for a series.

    Args:
        folder: Base directory containing the raster time series.
        time_points: Ordered list of years to read.
        class_name: Prefix used to build the raster file names.

    Returns:
        A 3-D array with shape (time, rows, cols) in int16 precision.
    """
    arrays: list[np.ndarray] = []
    for year in time_points:
        raster_path = folder / f"{class_name}{year}.tif"
        if not raster_path.exists():
            raise FileNotFoundError(f"Raster {raster_path} not found.")
        with rasterio.open(raster_path) as src:
            arrays.append(src.read(1))
    return np.stack(arrays).astype(np.int16)


def get_reference_meta(
    folder: Path,
    time_points: list[int],
    class_name: str,
) -> dict:
    """Copy the metadata from the first raster as template for outputs.

    Args:
        folder: Base directory containing raster files for X or Y series.
        time_points: Ordered list of years to read.
        class_name: Prefix used to build the raster file names.

    Returns:
        The metadata dictionary from the first raster.
    """
    reference = folder / f"{class_name}{time_points[0]}.tif"
    with rasterio.open(reference) as src:
        return src.meta.copy()


def write_raster(
    array: np.ndarray,
    reference_meta: dict,
    output_path: Path,
    dtype: np.dtype,
    nodata: int,
) -> None:
    """Persist a single-band raster using a metadata template.

    Args:
        array: The array to write to disk.
        reference_meta: Metadata copied from an input raster.
        output_path: Destination path for the GeoTIFF file.
        dtype: Desired numpy dtype for persistence.
        nodata: NoData sentinel value applied to the output raster.
    """
    meta = reference_meta.copy()
    meta.update(count=1, dtype=dtype, nodata=nodata, compress="lzw")
    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(array, 1)


def clip_to_dtype(
    array: np.ndarray,
    dtype: np.dtype,
) -> np.ndarray:
    """Clamp array values to the target dtype range and cast.

    Args:
        array: Array to clamp.
        dtype: Target dtype for the output array.

    Returns:
        Array cast to the provided dtype with safe range limits.
    """
    info = np.iinfo(dtype)
    return np.clip(array, info.min, info.max).astype(dtype)


def equation_49_hit_sum(
    x_stack: np.ndarray,
    y_stack: np.ndarray,
    nodata: int,
) -> np.ndarray:
    """Sum minima across time for pixels valid in both stacks.

    Args:
        x_stack: Raster time series for source X.
        y_stack: Raster time series for source Y.
        nodata: Sentinel value that marks invalid pixels.

    Returns:
        Array containing the hit sum per pixel.
    """
    valid = (x_stack != nodata) & (y_stack != nodata)
    hits = np.where(valid, np.minimum(x_stack, y_stack), 0)
    return hits.sum(axis=0)


def equation_50_difference_sum(
    x_stack: np.ndarray,
    y_stack: np.ndarray,
    nodata: int,
) -> np.ndarray:
    """Sum differences y - x over time ignoring NoData pixels.

    Args:
        x_stack: Raster time series for source X.
        y_stack: Raster time series for source Y.
        nodata: Sentinel value that marks invalid pixels.

    Returns:
        Signed difference sum per pixel.
    """
    valid = (x_stack != nodata) & (y_stack != nodata)
    diff = np.where(valid, y_stack - x_stack, 0)
    return diff.sum(axis=0)


def equation_51_temporal_allocation_difference(
    x_stack: np.ndarray,
    y_stack: np.ndarray,
    nodata: int,
) -> np.ndarray:
    """Compute the temporal allocation disagreement per pixel.

    Args:
        x_stack: Raster time series for source X.
        y_stack: Raster time series for source Y.
        nodata: Sentinel value that marks invalid pixels.

    Returns:
        Array with the temporal allocation difference values.
    """
    valid = (x_stack != nodata) & (y_stack != nodata)
    diff = np.where(valid, y_stack - x_stack, 0)
    abs_diff_sum = np.abs(diff).sum(axis=0)
    total_y = np.where(valid, y_stack, 0).sum(axis=0)
    total_x = np.where(valid, x_stack, 0).sum(axis=0)
    return abs_diff_sum - np.abs(total_y - total_x)


def equation_52_change_hits(
    x_stack: np.ndarray,
    y_stack: np.ndarray,
    nodata: int,
) -> np.ndarray:
    """Calculate signed change hits (gains minus losses).

    Args:
        x_stack: Raster time series for source X.
        y_stack: Raster time series for source Y.
        nodata: Sentinel value that marks invalid pixels.

    Returns:
        Array with net change hits per pixel.
    """
    dx = np.diff(x_stack, axis=0)
    dy = np.diff(y_stack, axis=0)
    valid = (
        (x_stack[:-1] != nodata)
        & (x_stack[1:] != nodata)
        & (y_stack[:-1] != nodata)
        & (y_stack[1:] != nodata)
    )
    loss_hit = np.where(valid & (dx < 0) & (dy < 0), np.maximum(dx, dy), 0)
    gain_hit = np.where(valid & (dx > 0) & (dy > 0), np.minimum(dx, dy), 0)
    return (gain_hit - loss_hit).sum(axis=0)


def equation_53_extent_difference(
    x_stack: np.ndarray,
    y_stack: np.ndarray,
    nodata: int,
) -> np.ndarray:
    """Compute extent difference between first and last time steps.

    Args:
        x_stack: Raster time series for source X.
        y_stack: Raster time series for source Y.
        nodata: Sentinel value that marks invalid pixels.

    Returns:
        Array with extent difference values per pixel.
    """
    valid = (
        (x_stack[0] != nodata)
        & (x_stack[-1] != nodata)
        & (y_stack[0] != nodata)
        & (y_stack[-1] != nodata)
    )
    delta_y = np.where(valid, y_stack[-1] - y_stack[0], 0)
    delta_x = np.where(valid, x_stack[-1] - x_stack[0], 0)
    return delta_y - delta_x


def equation_54_flexibility_difference(
    x_stack: np.ndarray,
    y_stack: np.ndarray,
    nodata: int,
    en_raw: np.ndarray,
) -> np.ndarray:
    """Compute flexibility difference using Equation 54.

    Args:
        x_stack: Raster time series for source X.
        y_stack: Raster time series for source Y.
        nodata: Sentinel value that marks invalid pixels.
        en_raw: Extent difference array used for the adjustment term.

    Returns:
        Array with flexibility difference values per pixel.
    """
    dy = np.diff(y_stack, axis=0)
    dx = np.diff(x_stack, axis=0)
    valid = (
        (x_stack[:-1] != nodata)
        & (x_stack[1:] != nodata)
        & (y_stack[:-1] != nodata)
        & (y_stack[1:] != nodata)
    )
    abs_diff = np.where(valid, np.abs(dy - dx), 0)
    sum_abs_diff = abs_diff.sum(axis=0)
    return sum_abs_diff - np.abs(en_raw)


In [ ]:
# Combined processing
print("Starting equations 49 to 54 processing...")
reference_meta = get_reference_meta(
    PATH_SERIES_X,
    TIME_POINTS,
    CLASS_NAME,
)
x_stack = read_stack(
    PATH_SERIES_X,
    TIME_POINTS,
    CLASS_NAME,
)
y_stack = read_stack(
    PATH_SERIES_Y,
    TIME_POINTS,
    CLASS_NAME,
)
en_raw = equation_53_extent_difference(
    x_stack,
    y_stack,
    NODATA_VALUE,
)
results_raw = {
    "Equation49": equation_49_hit_sum(
        x_stack,
        y_stack,
        NODATA_VALUE,
    ),
    "Equation50": equation_50_difference_sum(
        x_stack,
        y_stack,
        NODATA_VALUE,
    ),
    "Equation51": equation_51_temporal_allocation_difference(
        x_stack,
        y_stack,
        NODATA_VALUE,
    ),
    "Equation52": equation_52_change_hits(
        x_stack,
        y_stack,
        NODATA_VALUE,
    ),
    "Equation53": en_raw,
    "Equation54": equation_54_flexibility_difference(
        x_stack,
        y_stack,
        NODATA_VALUE,
        en_raw,
    ),
}
for name, array in results_raw.items():
    config = OUTPUT_DEFINITIONS[name]
    processed = clip_to_dtype(
        array,
        config["dtype"],
    )
    output_file = OUTPUT_PATH / config["filename"]
    write_raster(
        processed,
        reference_meta,
        output_file,
        config["dtype"],
        config["nodata"],
    )
    print(
        f"{name}: min={processed.min()} max={processed.max()} saved to {output_file}",
    )
print("Processing finished.")
